# Linear SVM for CIFAR-10 Image Classification

## What this notebook demonstrates

- CIFAR-10 image classification with a linear SVM
- structured multiclass hinge loss
- numerical and analytic gradients
- vectorized loss and gradient computation
- stochastic gradient descent and hyperparameter search

## Academic context

This notebook is a cleaned portfolio version of MSc coursework and implementation practice. It is included to demonstrate foundational computer vision and deep learning skills.

## Local helper package

This notebook uses the included project-local `engine/` package for coursework helper code such as data loading, classifiers, layers, optimization, and visualization utilities.


In [ ]:
# Project-local setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "engine").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the project root containing the engine/ package.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


### Set up code


In [ ]:
# Run some setup code for this notebook.
import random
import numpy as np
import matplotlib.pyplot as plt

# Display matplotlib figures inline.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Reload local helper modules during iterative notebook work.
%load_ext autoreload
%autoreload 2


### CIFAR-10 Data Loading and Pre-processing


In [ ]:
from engine.data_utils import load_CIFAR10

# Load the raw CIFAR-10 data.
cifar10_dir = str(DATA_DIR / 'cifar-10-batches-py')

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


Check the type of X arrays.


In [ ]:
print('The type of X_train array as loaded is {}'.format(X_train.dtype))
print('The type of X_test array as loaded is {}'.format(X_test.dtype))


Let's convert ``uint`` X arrays to ``float``. This is necessary for some computations that follow.


In [ ]:
# Convert arrays uint8 -> float
X_train = X_train.astype(float)
X_test = X_test.astype(float)


Check their type now!


In [ ]:
print('The type of X_train array as loaded is {}'.format(X_train.dtype))
print('The type of X_test array as loaded is {}'.format(X_test.dtype))


Once our full dataset is loaded, we choose to store it in ``.npy`` files as we did in ``knn.ipynb``.

Recall that by this way we can easily load the dataset, in case our runtime crashes.

Apart from that, loading from ``.npy`` files is most of the times faster than loading from image files like ``.jpeg`` or ``.png``, as the latter needs some extra steps.


In [ ]:
# First, let's create a new folder called arrays
# inside the data directory. Specify dir:
# NOTICE: This directory should already exist!
data_dir = str(DATA_DIR)
arrays_dir = os.path.join(data_dir, 'arrays')

# Create arrays directory
if not os.path.exists(arrays_dir):
    os.makedirs(arrays_dir)

# Save arrays in .npy files inside the new arrays dir
np.save(os.path.join(arrays_dir, 'full_X_train.npy'), X_train)
np.save(os.path.join(arrays_dir, 'full_y_train.npy'), y_train)
np.save(os.path.join(arrays_dir, 'full_X_test.npy'), X_test)
np.save(os.path.join(arrays_dir, 'full_y_test.npy'), y_test)

# Check that arrays are indeed saved
os.listdir(arrays_dir)


Now that we have our dataset stored in ``.npy`` files, we can use the following coding snippet to easily load it back if needed. Please remember to uncomment the following in case you want to load!


### Load .npy files


In [ ]:
# Specify data directory again, so that arrays dir is
# automatically retrieved and arrays can be loaded from it
data_dir = str(DATA_DIR)
arrays_dir = os.path.join(data_dir, 'arrays')

# Load arrays if needed. Uncomment to use!
X_train = np.load(os.path.join(arrays_dir, 'full_X_train.npy'))
y_train = np.load(os.path.join(arrays_dir, 'full_y_train.npy'))
X_test = np.load(os.path.join(arrays_dir, 'full_X_test.npy'))
y_test = np.load(os.path.join(arrays_dir, 'full_y_test.npy'))

# Num of samples for training and test
num_training = X_train.shape[0]
num_test = X_test.shape[0]


### Data visualization


In [ ]:
# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
classes = ['Plane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls[:7])
plt.show()


### Split Data


Let's split our data!


In [ ]:
# Split the data into train, val, and test sets. In addition we will
# create a small development set as a subset of the training data.
# We can use this for development so our code runs faster.
num_training = 45000
num_validation = 5000
num_test = 10000
num_dev = 500

# Our validation set will be num_validation points from the original
# training set.
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# Our training set will be the first num_train points from the original
# training set.
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# We will also make a development set, which is a small subset of
# the training set.
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# We use the first num_test points of the original test set as our
# test set.
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


Let's reshape the data into row vectors (flattening).


In [ ]:
# Preprocessing: reshape the image data into rows
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

# As a sanity check, print out the shapes of the data
print('Training data shape: ', X_train.shape)
print('Validation data shape: ', X_val.shape)
print('Test data shape: ', X_test.shape)
print('dev data shape: ', X_dev.shape)


### Pre-process data


Let's normalize our dataset.


In [ ]:
# Preprocessing: subtract the mean image
# first: compute the image mean based on the training data
mean_image = np.mean(X_train, axis=0)

# Visualize some of its elements and the image itself
print('Some of the elements of the mean image:')
print(mean_image[:10]) # print a few of the elements
print('The mean image of our train set:')
plt.figure(figsize=(4,4))
plt.imshow(mean_image.reshape((32,32,3)).astype('uint8')) # visualize the mean image
plt.show()

# second: subtract the mean image from train and test data
mean_image = mean_image.astype('uint8')
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

# third: append the bias dimension of ones (i.e. bias trick) so that our SVM
# only has to worry about optimizing a single weight matrix W
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])


Let's see our arrays' new shapes after bias trick.


In [ ]:
print(X_train.shape, X_val.shape, X_test.shape, X_dev.shape)


### SVM Classifier


#### Forward and Backward (naive)


**Implementation note.**

This section validates the forward pass and backward pass for the structured SVM loss function. The implementation lives in `svm_loss_naive()` in `engine/classifiers/linear_svm.py`. The forward pass should compute the SVM loss, and the backward pass should compute the gradient of the loss with respect to the weights.

To help you implement the gradient, we mention that:

$\nabla_{w_{y_{i}}} L_{i} = - \left( \sum_{j \neq i} \mathbf{1}(s_{j} - s_{y_{i}} + 1 > 0) \right) x_{i}$,

and

$\nabla_{w_{j}} L_{i} = \mathbf{1}(s_{j} - s_{y_{i}} + 1 > 0) x_{i}$,

where: $s_{j} = w_{j}^Tx_{i}$, $s_{y_{i}} = w_{y_{i}}^Tx_{i}$,

$\mathbf{1}$ is the indicator function that is $\bf{one}$ if the condition inside is $\bf{True}$, or $\bf{zero}$ if the condition is $\bf{False}$.


#### Gradient Check


To verify that your computed gradient is correct, we use a technique called numerical gradient checking. This process involves estimating the gradient of the loss function by slightly perturbing the weights and measuring the resulting change in the loss. Then, we compare this numerically estimated gradient with the gradient you computed analytically.

Here’s a breakdown of why and how we do this:

- **Analytic Gradient** (Backward Pass):
The analytic gradient, which you computed in the `svm_loss_naive` function, is derived using calculus directly on the loss function. This is efficient and exact (in theory), but it can be error-prone, especially in implementations with complex functions.
- **Numerical Gradient**:
The numerical gradient approximates the gradient by using a small value (called epsilon) to slightly change each weight in $W$ and observe the change in loss. For a weight $W_{ij}$, the numerical gradient is calculated as:

$\frac{\partial L}{\partial W_{ij}} \approx \frac{L(W + \epsilon) - L(W - \epsilon)}{2 \epsilon}$,

where $\epsilon$ is a small value (e.g., $10^{-5}$).

By comparing the analytic gradient to the numerical gradient, we check if our analytic calculations are correct. Small differences are expected due to numerical approximation errors, but if there are large discrepancies, it likely means there is an error in our analytic computation.


In [ ]:
# After defining the gradient, recompute it with the code below
# and gradient check it with the function we provided for you
from engine.classifiers import *

D = X_dev.shape[1]
C = np.max(y_dev) + 1

# Initialize W with small random values
W = np.ones((D, C)) * 0.001

# Compute the loss and its gradient at W.
loss, grad = svm_loss_naive(W, X_dev, y_dev, 0.0)

# Numerically compute the gradient along several randomly chosen dimensions, and
# compare them with your analytically computed gradient. The numbers should match
# almost exactly along all dimensions.
from engine.gradient_check import grad_check_sparse
f = lambda w: svm_loss_naive(w, X_dev, y_dev, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, grad)

# Do the gradient check once again with regularization turned on
# You didn't forget the regularization gradient did you?
loss, grad = svm_loss_naive(W, X_dev, y_dev, 5e1)
f = lambda w: svm_loss_naive(w, X_dev, y_dev, 5e1)[0]
grad_numerical = grad_check_sparse(f, W, grad)


#### Forward and Backward (vectorized)


**Implementation note.**

Implement the SVM loss function, but this time use vector/matrix products/multiplications instead of loops.

This vectorized version should be inside the ``svm_loss_vectorized()`` function of ``engine/classifiers/linear_svm.py``.

Inputs and outputs should be the same as in ``svm_loss_naive``.


In [ ]:
# Evaluate the vectorized SVM loss implementation.
import time

tic = time.time()
loss_naive, grad_naive = svm_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss: %e computed in %fs' % (loss_naive, toc - tic))

from engine.classifiers.linear_svm import svm_loss_vectorized
tic = time.time()
loss_vectorized, _ = svm_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss: %e computed in %fs' % (loss_vectorized, toc - tic))

# The losses should match but your vectorized implementation should be much faster.
print('difference: %f' % (loss_naive - loss_vectorized))


**Implementation note.**

Validate the implementation of ``svm_loss_vectorized()`` by computing the gradient of the loss function in a vectorized way.


In [ ]:
# Compute the vectorized SVM gradient
# of the loss function in a vectorized way.

# The naive implementation and the vectorized implementation should match, but
# the vectorized version should still be much faster.
tic = time.time()
loss_naive, grad_naive = svm_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print(f'Naive SVM computed in {(toc - tic)}')

tic = time.time()
loss_vectorized, grad_vectorized = svm_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print(f'Vectorized SVM computed in {(toc - tic)}')


# The loss is a single number, so it is easy to compare the values computed
# by the two implementations. The gradient on the other hand is a matrix, so
# we use the Frobenius norm to compare them.
print('difference in gradients: %f' % np.linalg.norm(grad_naive - grad_vectorized, ord='fro'))
print('difference in loss: %f' % np.linalg.norm(loss_naive - loss_vectorized))


#### Stochastic Gradient Descent


We now have vectorized and efficient expressions for the loss and the gradient.

Our gradient matches the numerical gradient.

We are therefore ready to use Stochastic Gradient Descent (SGD) to minimize the loss.


**Implementation note.**
This section uses the SGD implementation in `engine/classifiers/linear_classifier.py` inside ``LinearClassifier.train()``.


In [ ]:
# Use the SGD implementation in linear_classifier.py.
# LinearClassifier.train() and then run it with the code below.
from engine.classifiers import LinearSVM
svm = LinearSVM()
tic = time.time()
loss_hist = svm.train(X_train, y_train, learning_rate=1e-10, reg=1e-4,
                      num_iters=1500, verbose=True)
toc = time.time()
print('That took %fs' % (toc - tic))


In [ ]:
# A useful debugging strategy is to plot the loss as a function of
# iteration number:
plt.plot(loss_hist)
plt.xlabel('Iteration number')
plt.ylabel('Loss value')
plt.show()


#### Predict


**Implementation note.**

This section uses the ``LinearClassifier.predict()`` function in `engine/classifiers/linear_classifier.py` to evaluate the performance of the model.


In [ ]:
# Write the LinearClassification.predict function
# and evaluate the performance on both the training and validation set
y_train_pred = svm.predict(X_train)
print('training accuracy: %f' % (np.mean(y_train == y_train_pred), ))
y_val_pred = svm.predict(X_val)
print('validation accuracy: %f' % (np.mean(y_val == y_val_pred), ))


#### Hyperparameter search


**Implementation note.**

Write below code that chooses the best hyperparameters by tuning on the validation set (grid search).

Your goal is to perform a real-world training in order to save the SVM mobel that performs the best on validation set.


In [ ]:
# Use the validation set to tune hyperparameters (regularization strength and
# learning rate). Expected: experiment with different ranges for the learning
# rates and regularization strengths; careful tuning can
# get a classification accuracy of about 0.39 on the validation set.

# Note: you may see runtime/overflow warnings during hyper-parameter search.
# This may be caused by extreme values, and is not a bug.

# results is dictionary mapping tuples of the form
# (learning_rate, regularization_strength) to tuples of the form
# (training_accuracy, validation_accuracy). The accuracy is simply the fraction
# of data points that are correctly classified.
results = {}
best_val = -1   # The highest validation accuracy that we have seen so far.
best_svm = None # The LinearSVM object that achieved the highest validation rate.

# Write code that chooses the best hyperparameters by tuning on the validation #
# set. For each combination of hyperparameters, train a linear SVM on the      #
# training set, compute its accuracy on the training and validation sets, and  #
# store these numbers in the results dictionary. In addition, store the best   #
# validation accuracy in best_val and the LinearSVM object that achieves this  #
# accuracy in best_svm.                                                        #
#                                                                              #
# Hint: Expected: use a small value for num_iters as you develop your         #
# validation code so that the SVMs don't take much time to train; once you are #
# confident that the validation code works, rerun the validation   #
# code with a larger value for num_iters.                                      #

# Provided as a reference. You may or may not want to change these hyperparameters
learning_rates = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
regularization_strengths = [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

# Obtain all possible combinations
grid_search = [(lr,rg) for lr in learning_rates for rg in regularization_strengths]

for lr, rg in grid_search:

    svm = LinearSVM()

    train_loss = svm.train(X_train, y_train, learning_rate=lr, reg=rg,
                      num_iters=1500, verbose=False)

    y_train_pred = svm.predict(X_train)
    train_accuracy = np.mean(y_train_pred == y_train)
    y_val_pred = svm.predict(X_val)

    val_accuracy = np.mean(y_val_pred == y_val)

    results[(lr,rg)] = (train_accuracy, val_accuracy)
    if best_val < val_accuracy:
        best_val = val_accuracy
        best_svm = svm

# Print out results.
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved during validation: %f' % best_val)


#### Visualization of search results


In [ ]:
# Visualize the results
import math

x_scatter = [math.log10(x[0]) for x in results]
y_scatter = [math.log10(x[1]) for x in results]

# plot training accuracy
marker_size = 100
colors = [results[x][0] for x in results]
plt.subplot(2, 1, 1)
plt.tight_layout(pad=3)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 training accuracy')

# plot validation accuracy
colors = [results[x][1] for x in results] # default size of markers is 20
plt.subplot(2, 1, 2)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 validation accuracy')
plt.show()


#### Predict on test set (unseen images)


Let's see how our model performs on unseen images - the test set.


In [ ]:
# Evaluate the best svm on test set
y_test_pred = best_svm.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print('Linear SVM on raw pixels final test set accuracy: %f' % test_accuracy)


#### Visualization of learned weights for each class


In [ ]:
# Visualize the learned weights for each class.
# Depending on your choice of learning rate and regularization strength, these may
# or may not be nice to look at.
w = best_svm.W[:-1,:] # strip out the bias
w = w.reshape(32, 32, 3, 10)
w_min, w_max = np.min(w), np.max(w)
classes = ['Plane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
for i in range(10):
    plt.subplot(2, 5, i + 1)

    # Rescale the weights to be between 0 and 255
    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])


**Concept check.**

Describe what your visualized SVM weights look like, and offer a brief explanation for why they look they way that they do.

**Answer.** The weights visualization should resemble a blurred shape of each class object. But, in contrast, we get a very noisy pattern, possibly due to the low accuracy of our model caused by the poor representation of the images. Regions that have more contrast appear more prominent and correspond to the parts of the image that the SVM finds most discriminative for separating a given class from others.
